# 01 — Data Overview
**Phase 1 · Multi-fault Diagnosis of Rotating Machinery**

Goals:
- Inspect the MaFaulDa dataset folder structure and file count per class
- Preview raw signals from each fault condition
- Check class balance and sampling consistency

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

In [19]:
DATA_ROOT = Path("/kaggle/input/datasets/vuxuancu/mafaulda-full/mafaulda")
SAMPLING_RATE = 50_000  # Hz

CHANNEL_NAMES = [
    "tachometer",
    "underhang_axial", "underhang_radial", "underhang_tangential",
    "overhang_axial",  "overhang_radial",  "overhang_tangential",
    "microphone",
]

In [20]:
records = []

for csv_path in sorted(DATA_ROOT.rglob("*.csv")):
    parts = csv_path.relative_to(DATA_ROOT).parts
    # parts looks like one of these:
    #   ('normal', '12.288.csv')
    #   ('imbalance', '6g', '12.288.csv')
    #   ('horizontal-misalignment', '1.0mm', '12.288.csv')
    #   ('underhang', 'ball_fault', '20g', '12.288.csv')
    #   ('overhang', 'outer_race', '35g', '12.288.csv')

    fault_category = parts[0]
    filename = parts[-1]
    rotation_hz = float(filename.replace(".csv", ""))
    rotation_rpm = rotation_hz * 60
    frequency_hz = round(rotation_hz)

    # Defaults — will be overridden per category below.
    sub_type = None
    bearing_position = None
    severity = None
    severity_value = None
    severity_unit = None

    if fault_category == "normal":
        pass  # all defaults are fine

    elif fault_category == "imbalance":
        severity = parts[1]                          # e.g., '6g'
        severity_value = float(severity.replace("g", ""))
        severity_unit = "g"

    elif fault_category in ("horizontal-misalignment", "vertical-misalignment"):
        severity = parts[1]                          # e.g., '1.0mm'
        severity_value = float(severity.replace("mm", ""))
        severity_unit = "mm"

    elif fault_category in ("underhang", "overhang"):
        bearing_position = fault_category
        sub_type = parts[1]                          # 'ball_fault', 'cage_fault', 'outer_race'
        severity = parts[2]                          # e.g., '20g'
        severity_value = float(severity.replace("g", ""))
        severity_unit = "g"

    records.append({
        "file_path": csv_path,
        "fault_category": fault_category,
        "sub_type": sub_type,
        "bearing_position": bearing_position,
        "severity": severity,
        "severity_value": severity_value,
        "severity_unit": severity_unit,
        "frequency_hz": frequency_hz,
        "rotation_hz": rotation_hz,
        "rotation_rpm": rotation_rpm,
    })

metadata = pd.DataFrame(records)
metadata.to_csv("metadata.csv", index=False)

print(f"Total files: {len(metadata)}")
print(metadata["fault_category"].value_counts())
metadata.head()

In [30]:
FAULT_TYPES = metadata["fault_category"].unique()
CHANNEL     = "underhang_axial"
N_SAMPLES   = 5_000  # first 0.1 s at 50 kHz — enough to see waveform character

fig, axes = plt.subplots(len(FAULT_TYPES), 1, figsize=(14, 3 * len(FAULT_TYPES)), sharex=True)
t = np.arange(N_SAMPLES) / SAMPLING_RATE * 1000  # ms

for ax, fault in zip(axes, FAULT_TYPES):
    sample_path = metadata.loc[metadata["fault_category"] == fault, "file_path"].iloc[0]
    df = pd.read_csv(sample_path, header=None, names=CHANNEL_NAMES)
    
    severity = metadata.loc[metadata["fault_category"] == fault, "severity"].iloc[0]
    label    = fault if severity is None else f"{fault}  ({severity})"

    ax.plot(t, df[CHANNEL].values[:N_SAMPLES], linewidth=0.6)
    ax.set_ylabel("Accel (g)", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle(f"Representative signal — {CHANNEL}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [31]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(metadata["frequency_hz"], bins=50, edgecolor="white", linewidth=0.4)

ax.set_xlabel("Rotation frequency (Hz)")
ax.set_ylabel("Number of files")
ax.set_title("Distribution of rotation speeds across MAFAULDA")

plt.tight_layout()

plt.show()

In [32]:
ACCEL_CHANNELS = [
    "underhang_axial", "underhang_radial", "underhang_tangential",
    "overhang_axial",  "overhang_radial",  "overhang_tangential",
]

FAULT_TYPES = metadata["fault_category"].unique()
CHANNEL     = "underhang_axial"
# CHANNEL_NAMES
N_SAMPLES   = 5_000  # first 0.1 s at 50 kHz — enough to see waveform character

fig, axes = plt.subplots(len(ACCEL_CHANNELS), 1, figsize=(14, 3 * len(ACCEL_CHANNELS)), sharex=True)

sample_path = metadata.loc[metadata["fault_category"] == "normal", "file_path"].iloc[0]
df = pd.read_csv(sample_path, header=None, names=CHANNEL_NAMES)

fs = 50_000  # Hz — MAFAULDA's sampling rate
n_samples = len(df)
time = np.arange(n_samples) / fs * 1000 # miliseconds
window_samples = int(2/1000 * fs)  # 5000    0.01 = 10 ms; 0.001 = 1 ms

for ax, accel_ch in zip(axes, ACCEL_CHANNELS):
    label    =  f"{accel_ch}"
    ax.plot(time[:window_samples], df[accel_ch].values[:window_samples])
    ax.set_ylabel("Acceleration (g)", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle(f"Representative signal — Normal", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [33]:
sample_path

In [34]:
fig_fs, axes_fs = plt.subplots(len(ACCEL_CHANNELS), 1, figsize=(14, 3 * len(ACCEL_CHANNELS)), sharex=True)

fault_sample_path = metadata.loc[metadata["fault_category"] == "vertical-misalignment", "file_path"].iloc[0]
df_fs = pd.read_csv(fault_sample_path, header=None, names=CHANNEL_NAMES)

for ax, accel_ch in zip(axes_fs, ACCEL_CHANNELS):
    label    =  f"{accel_ch}"
    ax.plot(time[:window_samples], df_fs[accel_ch].values[:window_samples])
    ax.set_ylabel("Acceleration (g)", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)

axes_fs[-1].set_xlabel("Time (ms)")
fig.suptitle(f"Representative signal — Normal", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [35]:
fault_sample_path

In [36]:
# Create the plot for "normal" sample (underhang)   
plt.figure(figsize=(14, 3))
plt.plot(time[:window_samples], df[ACCEL_CHANNELS[0]].values[:window_samples], label='acc axial')
plt.plot(time[:window_samples], df[ACCEL_CHANNELS[1]].values[:window_samples], label='acc rad') 
plt.plot(time[:window_samples], df[ACCEL_CHANNELS[2]].values[:window_samples], label='acc tan')
plt.legend(loc='upper right')

# Add details
plt.title('Normal')
plt.xlabel("Time (ms)")
plt.ylabel("Sensor Value (m/s²)")
plt.ylim(-5,5)
plt.grid(alpha=0.3)
plt.show()

In [37]:
fig_test, axes_test = plt.subplots(1, 2, figsize=(14, 3))

# Create the plot for "normal" sample (underhang)   
# axes_test[0].figure(figsize=(14, 3))
axes_test[0].plot(time[:window_samples], df[ACCEL_CHANNELS[0]].values[:window_samples], label='acc axial')
axes_test[0].plot(time[:window_samples], df[ACCEL_CHANNELS[1]].values[:window_samples], label='acc rad') 
axes_test[0].plot(time[:window_samples], df[ACCEL_CHANNELS[2]].values[:window_samples], label='acc tan')
axes_test[0].legend(loc='upper right')

# Add details
axes_test[0].set_title('Normal')
axes_test[0].set_xlabel("Time (ms)")
axes_test[0].set_ylabel("Sensor Value (m/s²)")
axes_test[0].set_ylim(-5,5)
axes_test[0].grid(alpha=0.3)

# Create the plot for "vertical-misalignment" sample (underhang)   
# axes_test[0].figure(figsize=(14, 3))
axes_test[1].plot(time[:window_samples], df_fs[ACCEL_CHANNELS[0]].values[:window_samples], label='acc axial')
axes_test[1].plot(time[:window_samples], df_fs[ACCEL_CHANNELS[1]].values[:window_samples], label='acc rad') 
axes_test[1].plot(time[:window_samples], df_fs[ACCEL_CHANNELS[2]].values[:window_samples], label='acc tan')
axes_test[1].legend(loc='upper right')

# Add details
axes_test[1].set_title('Vertical Misalignment')
axes_test[1].set_xlabel("Time (ms)")
axes_test[1].set_ylabel("Sensor Value (m/s²)")
axes_test[1].set_ylim(-6,6)
axes_test[1].grid(alpha=0.3)

plt.show()

In [38]:
# Raw underhang accelerometer signals of healthy and faulty machine
# subtypes for under-over/hang  --> 'ball_fault', 'cage_fault', 'outer_race'
FAULT_SUB_TYPES = ['ball_fault', 'cage_fault', 'outer_race']
TARGET_FREQ = 22  # Hz 

dfs = {}
for cat in FAULT_TYPES:
    
    if cat=='underhang' or cat=='overhang':
        for sub_cat in FAULT_SUB_TYPES:
            path = metadata.loc[(metadata["fault_category"] == cat) & 
                                (metadata["sub_type"] == sub_cat) &
                                (metadata["frequency_hz"] == TARGET_FREQ) , "file_path"].iloc[0]
            dfs[f'{cat}_{sub_cat}'] = pd.read_csv(path, header=None, names=CHANNEL_NAMES)
            
    else:
        path = metadata.loc[(metadata["fault_category"] == cat) &
                            (metadata["frequency_hz"] == TARGET_FREQ), "file_path"].iloc[0]
    dfs[cat] = pd.read_csv(path, header=None, names=CHANNEL_NAMES)

fig_uh_acc, axes_uh_acc = plt.subplots(5, 2, figsize=(14, 3*5))
axes_uh_acc = axes_uh_acc.flatten()

panels = [
    (axes_uh_acc[0], dfs["normal"], "Normal"),
    (axes_uh_acc[1], dfs["imbalance"], "Imbalance"),
    (axes_uh_acc[2], dfs["vertical-misalignment"], "Vertical Misalignment"),
    (axes_uh_acc[3], dfs["horizontal-misalignment"], "Horizontal Misalignment"),
    (axes_uh_acc[4], dfs["underhang_ball_fault"], "underhang_ball_fault"),
    (axes_uh_acc[5], dfs["underhang_cage_fault"], "underhang_cage_fault"),
    (axes_uh_acc[6], dfs["underhang_outer_race"], "underhang_outer_race"),
    (axes_uh_acc[7], dfs["overhang_ball_fault"], "overhang_ball_fault"),
    (axes_uh_acc[8], dfs["overhang_cage_fault"], "overhang_cage_fault"),
    (axes_uh_acc[9], dfs["overhang_outer_race"], "overhang_outer_race")
]


for ax, data, title in panels:
    ax.plot(time[:window_samples], data[ACCEL_CHANNELS[0]].values[:window_samples], label="acc axial")
    ax.plot(time[:window_samples], data[ACCEL_CHANNELS[1]].values[:window_samples], label="acc rad")
    ax.plot(time[:window_samples], data[ACCEL_CHANNELS[2]].values[:window_samples], label="acc tan")
    ax.legend(loc="upper right")
    ax.set_title(title)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Sensor Value (m/s²)")
    ax.set_ylim(-6, 6)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [39]:
FAULT_TYPES[2:4]

In [40]:
# Raw sensor signals of healthy machine and machine with specific fault (bearing ball) 
# specific time variables for better vizualization of Tachometer plot
time_tach = np.arange(n_samples) / fs # seconds
window_samples_tach = int(200/1000 * fs)

dfs_spec = {}
SPECIFIC_TYPES = [FAULT_TYPES[2],FAULT_TYPES[3]] # pick normal and overhang type fault
SPECIFIC_SUB_TYPE = FAULT_SUB_TYPES[0] # bearing ball fault
for cat in SPECIFIC_TYPES:
    if cat=='underhang' or cat=='overhang':
        path = metadata.loc[(metadata["fault_category"] == cat) & 
                            (metadata["sub_type"] == SPECIFIC_SUB_TYPE) &
                            (metadata["frequency_hz"] == TARGET_FREQ) , "file_path"].iloc[0]
        dfs_spec[f'{cat}_{SPECIFIC_SUB_TYPE}'] = pd.read_csv(path, header=None, names=CHANNEL_NAMES)
            
    else:
        path = metadata.loc[(metadata["fault_category"] == cat) &
                            (metadata["frequency_hz"] == TARGET_FREQ), "file_path"].iloc[0]
        dfs_spec[cat] = pd.read_csv(path, header=None, names=CHANNEL_NAMES)

fig_spec, axes_spec = plt.subplots(4, 2, figsize=(14, 3*4))
axes_spec = axes_spec.flatten()

panels = [
    (axes_spec[0], dfs_spec["normal"], "Underhang Acc - Normal"),
    (axes_spec[1], dfs_spec["overhang_ball_fault"], "Underhang Acc - Ball Fault"),
    (axes_spec[2], dfs_spec["normal"], "Overhang Acc - Normal"),
    (axes_spec[3], dfs_spec["overhang_ball_fault"], "Overhang Acc - Ball Fault"),
    (axes_spec[4], dfs_spec["normal"], "Tachometer - Normal"),
    (axes_spec[5], dfs_spec["overhang_ball_fault"], "Tachometer - Ball Fault"),
    (axes_spec[6], dfs_spec["normal"], "Microphone - Normal"),
    (axes_spec[7], dfs_spec["overhang_ball_fault"], "Microphone - Ball Fault")
]


for i , (ax, data, title) in enumerate(panels):
    # define key to select the correct values for each subplot
    if i<=3:
        if i<= 1:
            key = 0
        else:
            key = 3
        ax.plot(time[:window_samples], data[ACCEL_CHANNELS[key]].values[:window_samples], label="acc axial")
        ax.plot(time[:window_samples], data[ACCEL_CHANNELS[key+1]].values[:window_samples], label="acc rad")
        ax.plot(time[:window_samples], data[ACCEL_CHANNELS[key+2]].values[:window_samples], label="acc tan")
        ax.legend(loc="upper right")
        ax.set_ylabel("Sensor Value (m/s²)")
        ax.set_ylim(-6, 6)
        
    elif i==4 or i==5: # tachometer
        ax.plot(time_tach[:window_samples_tach], data[CHANNEL_NAMES[0]].values[:window_samples_tach])
        ax.set_ylabel("Sensor Value (V)")
        ax.set_ylim(-1.5, 5.5)
        # Set X-axis to 1 decimal place
        ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    else:
        ax.plot(time[:window_samples], data[CHANNEL_NAMES[7]].values[:window_samples], color='red')
        ax.set_ylabel("Sensor Value (Pa)")
        ax.set_ylim(-0.3, 0.3)
    
    ax.set_title(title)
    ax.set_xlabel("Time (ms)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [41]:
8*9

In [55]:
8+8